#use llamafactory environment

In [1]:
from datasets import load_dataset
import pandas as pd
import json


In [2]:
#load training set of hellaswag
dataset = load_dataset("allenai/ai2_arc", name="ARC-Easy", split="train")

#print dataset info
print("Dataset info:")
print(dataset)

print("# samples:", len(dataset)) #should be 39k
n_samples = len(dataset)

#convert to dataframe
df = dataset.to_pandas()
df.head()


Dataset info:
Dataset({
    features: ['id', 'question', 'choices', 'answerKey'],
    num_rows: 2251
})
# samples: 2251


,id,question,choices,answerKey
0,Mercury_7220990,Which factor will most likely cause a person t...,{'text': ['a leg muscle relaxing after exercis...,B
1,MCAS_2007_8_5189,Lichens are symbiotic organisms made of green ...,"{'text': ['carbon dioxide', 'food', 'protectio...",B
2,Mercury_SC_401169,When a switch is used in an electrical circuit...,"{'text': ['cause the charge to build.', 'incre...",D
3,MCAS_2004_8_27,Which of the following is an example of an ass...,"{'text': ['contact lens', 'motorcycle', 'rainc...",A
4,NYSEDREGENTS_2006_8_10,"Rocks are classified as igneous, metamorphic, ...","{'text': ['their color', 'their shape', 'how t...",3


In [3]:
# Add an 'answer' column by matching answerKey to the corresponding choice text.
def extract_answer(row):
    labels = row["choices"]["label"]
    texts = row["choices"]["text"]
    label_to_text = dict(zip(labels, texts))
    return label_to_text.get(row["answerKey"], None)

df["answer"] = df.apply(extract_answer, axis=1)
df.head(100)

,id,question,choices,answerKey,answer
0,Mercury_7220990,Which factor will most likely cause a person t...,{'text': ['a leg muscle relaxing after exercis...,B,a bacterial population in the bloodstream
1,MCAS_2007_8_5189,Lichens are symbiotic organisms made of green ...,"{'text': ['carbon dioxide', 'food', 'protectio...",B,food
2,Mercury_SC_401169,When a switch is used in an electrical circuit...,"{'text': ['cause the charge to build.', 'incre...",D,stop and start the flow of current.
3,MCAS_2004_8_27,Which of the following is an example of an ass...,"{'text': ['contact lens', 'motorcycle', 'rainc...",A,contact lens
4,NYSEDREGENTS_2006_8_10,"Rocks are classified as igneous, metamorphic, ...","{'text': ['their color', 'their shape', 'how t...",3,how they formed
...,...,...,...,...,...
95,Mercury_7081428,When an earthquake wave passes from the crust ...,"{'text': ['reverses direction.', 'changes spee...",B,changes speed.
96,OHAT_2008_5_20,A student wraps a wire around an iron nail. Th...,"{'text': ['gravitational force', 'magnetic for...",B,magnetic force
97,MSA_2015_5_8,Natural processes cause rapid and slow changes...,"{'text': ['an earthquake shaking the ground', ...",A,an earthquake shaking the ground
98,Mercury_7219135,"Since 1961, Nevada has led the United States i...","{'text': ['gold', 'uranium', 'lumber', 'iron']...",A,gold


In [4]:
### create json file

#format data for sft
data = []

for idx, row in df.iterrows():
    item = {
        "instruction": row["question"],
        "input": "",
        "output": row["answer"]
    }
    data.append(item)
    #track progress
    if idx % 10000 == 0:
        print(f"Processed {idx} rows")
    if idx == (n_samples - 1):
        print(f"Processed {idx} rows (last row)")

#save to JSON file
with open("data/arc_easy.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

print("Complete!")

Processed 0 rows
Processed 2250 rows (last row)
Complete!
